# ⚡ Google Drive GPU Refiner - Remote Worker Node (Helper)

This lightweight helper notebook provides external GPU compute (NVIDIA Tesla T4 / V100 / A100 / L4) to accelerate video refining for the **Main Google Drive Downloader & GPU Refiner**.

---

### 🌟 How it Works:
1. **Shared Drive Integration**: Mounts the shared `downloader` Google Drive folder.
2. **Zero Configuration**: Automatically discovers the Main coordinator via `data/cluster/master.json` on Google Drive.
3. **Hardware NVENC Acceleration**: Receives 3–5 minute video chunks, transcodes them at maximum GPU speed using NVENC hardware acceleration, and saves the finished segments directly back into Google Drive.
4. **Fault Tolerant**: If this Colab runtime stops or is preempted, the Main coordinator automatically reclaims the chunk and reassigns it to another worker without losing any completed work.


In [ ]:
#@title 🛠️ Step 1: Environment Setup & Google Drive Mount
#@markdown Mounts Google Drive, verifies NVIDIA GPU hardware acceleration, and prepares workspace.

import os
import sys
import shutil
import subprocess
import json
import base64
import zlib

print("=" * 72)
print("🚀 [1/3] Checking Hardware Acceleration (NVIDIA GPU)...")
print("=" * 72)
has_gpu = False
try:
    gpu_info = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"],
        text=True
    ).strip()
    print(f"✅ Active GPU Detected: {gpu_info}")
    has_gpu = True
except Exception:
    print("⚠️ No NVIDIA GPU detected! Helper will run in CPU fallback mode (slower).")
    print("👉 Recommendation: Click Runtime > Change runtime type > Select T4 GPU.")

print("\n" + "=" * 72)
print("📁 [2/3] Mounting Google Drive...")
print("=" * 72)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ Google Drive mounted successfully at /content/drive")
except Exception as e:
    print(f"⚠️ Drive mount notice: {e}")

print("\n" + "=" * 72)
print("📦 [3/3] Deploying Helper Modules...")
print("=" * 72)
workspace_dir = "/content/colab-gpu-helper"
os.makedirs(workspace_dir, exist_ok=True)
os.chdir(workspace_dir)
if workspace_dir not in sys.path:
    sys.path.insert(0, workspace_dir)

# Decompress and write modular files
bundle_b64 = "eJzdfQtb40ay6F/pMDeLnRgD89rEOeQcBjwzbHjMwTBz9sNcfbLcBgVb8uoB4+Xw3289uqVuqWWbSfa+lC+DJfWjuqq6qrqquvW4EcTRJLzpzhcbPbERzuZxkok4HUbqZyKLn3kejoubLJyVb9LbPAun5bvbRPrjMLoZRpMknolsMYcbod6ezbMwjvxpRxyHadYRh2EA/+5HC1V67me303Cki3+C22E0jF6IQ5nJIBP96D5M4mgmo6wnPsTxzVSKg3jqj8R9Ko7jwAdAjgbewdnx/juxB2PpYotd+RV6S1vDjW0YcQaVhxtt4UdjEcVZvdBBb4jXRhu7HssJ4CGNp/fSG8cP0TT2xzLxxmHSaoutX0WaJb1hJOAabtB/9PtwEfmzEMCZLsQUwMpkqsE9TMJ7KcqmBDQFQ4uTRUek+RzHjQhLb+FHkGdph1vs6gdbmZ/cyCzdGi22wjHDliNWOzSi9NZP5FhM4im0nYqH2xD6vMn9xIeBS2x5Ak9SbhSKitP+5/65eEjCDBCjgAGos9gGOInjrFsfaDghHGqcK1TgNfJTiWgyyOCPUvzb0vfwFtAkW56HMHleu11WT2SWJ1FR9fc4jFq6yQ6AYNAizfwbZDgmGFZ+IXa74lMSzvxkIYJbGdz1GhHYsoapCwlVSMQRM5gCTT32EB0wspKjtsdYf7uhFwNdFXYzG2wb6MuShXGH1yROYOaNRUhYmUJtZMKm+kU7HvZmEIEwadbqYKvtek1FWl0vTKk7as7VEV6IizDKpQOM2nD0BbhLQpgee+JKdqfxg8SJhWOV1ZFy19fuZgBaP1q07rCSbhJboQdXwC9+5g83kHMS/0H9kJMwkkCb67aAklhddmU0Th9CYNLhRjecL6IRiooCHNVy0/jxUozL0LqLpfmoRhAqbjM2MXTDWCtcBC2yQLOphY9Xgwql6mXk10DOQeDSH5Aua1N8RcW5n6blNH3ZFQc4PUXgR3GEAhN/jUOgliwEY6jFVfGKuKVstzYLTxY0m7dNXHaeVV5siYGaycurDkje0u/U6o/rKG5FBgqQgcohGHip0zOoks09f0DVTg1OwlsSsIF7OlNxk0tUb/jcxT70vIHwBsxarHMzKDNwylXfNiEduLxTa6paGApdr2ZkBMAutAYXM0M2DI3xUKOG0WdQMvOrrhhIPwluhXukxAUgZZRyBtPBZ12P8kZslsyzyS3OFlTdpWYK3DWpFVV3tUZBibaoSFpH5aJKPp1686r0UhU63Jqb+eiVFvBdGrsWtJbIayBzswJhYrgnAoPbIEYVYH/mrDCa/b9ndpRIapgl+lpT5uNVnzWOoTO+FA1WT18u+I2a5HUx+Vgisz2nYEwNIe2aTqYQb55TZivfNrGaWijqOWeXWetPmWIOpl5OpAYC6QsMiDrUXHgNi8ZpzQAsbntmJZBU5huZ6FBO/HyaiQks3Ea+ZZaU1pxaDY6XyGWHIWDUqksGftUuGK8o/K8QFc5CFRHy/DHqUSoRw9Xx8eHZl9Pjs/3D/rl3eHQODTatpocRjLyykDw+O9g/9r6cnf82+LR/0LfByeRsjl3LaSpV+Xf7g77q5rmLznpXFjvrlpGhCycC3jAUbXZTDLIkDwAFgDlrWXlomrLn+18qMFIPNqKKlQq0fN5/f3TaP1yzkl7VGBVPzj4f9Qeu+kbTWHkW3yOIZt2Lz2vUy+69VCbVqmcXHwuqL6sdZ7dKNhzuX+yvN0xez0GVk/7FPlX72+DstFZPtUdjk5mPtbq/p3FUUKw/v5UzmcAMJ/eR+HxiE4vZooFkFZ5RNPNCbl5VpbGuWxuJp30ZF+f7g4/e4GL/w9HphzUbyBI/vbU8IlyoCbGOJgrM8rul+LUbd2P5on/ySXVd6Q3ofX70WePGprFy+4E+SMJRnuGE+nQpDqZ5moHp/BdYNubRXQoTjkZq02wJyuqMZOLq4OPl6W+DZSg3XpEkICgMbKkW1sO0Ufvg+HJw4Z4sJnYDHr9VBZvsnzv7NFrF2g9xcicTu8OTffrz/ui4JvIqtWc+dl3SVQ24GfA6c5jg29WXDKLWT3UodoElA3K2VB1Wle00yx2QrzxPfLQiFKt4h5fn+xdHZ6feoH8Avb18vdPdKWwLMErFLIxy9FfMoYU7uZgkoHpEKm/Qgy1aMZgkMxA6I3/qR4EU8USwAZLFU5BJ+OgvAuw5pUXeXw764mj7THOqd9xHrXRxdNI/u7xQMLzaUTC8EG+K7qfST6UYK+hFa57Ie4AgRUsHXvgjsLXYrZ5iKZxRr7dm6K9AFgUL14/SIB6D1TqMmFTex/7++cW7/v5Fpf/Xb6D7F+LET+4EUwmGNZmCTmKPorgFMz0bST8jM/n1G5iwJ/v/5e0fXKAs0NT529m7AWK0ROYBu5riHLAZziW22BMwkK1RPpmgrzuSXzPxezzaTBnsVKQxSQ3FLFDgHpfdISzBxlPJpP4sk1EMKDiOb0iQ/AWMwFF+I07jLJyA5Yf4SsUFvMUah/13lx+847MPg54YxTGuOdGwkByW6N7IDIyp/un+u2OQakVZNhR20fL+Di2YHWK0wd9hoCdYgJi1h6GEZXM/XQAfzsC8vyn5FAQtOsUpUJL2bJtDR0Ro+Di0IJ4vtMj9z8v+JYD4blmH/8hlLrvjkTHJmqrVp7pVGWE9Uo70jvCnD/4iFTnAPSUAB4NDMfjP4zCTFJzA2AOQF9gPvYDWmGgGYJktqEmDmvjhFOyt9BdBVsRDCK1iy/YwXbalMRZ7cJZFaZSqNknmQ5RC78BZgUxTjDYYTktBy5lhhGzOprwKDlV0kDIu4IdpFtVtN+WGtA2zsmBhbXVESQhLX9XVGzwzpaFDGnec1nFHWHZRcVvYOh1Rs150oRK4qsosnpgwNeqJYaRXLPYCHBhz5t9JIETaGneYCF58t3eR5FLZ+ksWhXpBSPImyJNERsFCDOTMn9/GwGcgIL6wVDsGDs2U8NIk1fChHOY3x2dfvMGnPlK1XmYH5doOsMY45Bjh5enx0cnRRf8QHevce4YKYCudS9BFesmkelXoLttrkKMUx1RVwU5XQ4HyRbC2Wwyw5RoOoA2A8AgIz9nSaQzyHQaDf3sij3D6ovYMDCTiPEhxMKDsYJInYXA3leaYoDAI6BQIshaU9vCVmLkIZzLOM3IeDAhpX/wsuB3HN+I8p8BnMTjgzfOq9voZlKdRpKTexcfz/uDj2fGh9+4Tonp3Z0f8AP++fN0lKuL9b++2U3dlq4/dl9SJhWiG5bx/cQ6TnfUeiswwC0E+/gi3sLJWYbJXIB8zdFlmuPTMUi1et2ZyFoMEuvDTO3GR+Eo+AtLR47eFQXsxRSE6i6HdOKEYfQaFKZpyrX57KFgtpIMSucPVOb72gjiP0BoC3uV+P0zjkdIx8BhXxkk8TQ1+4zeeH2Tsy8ZpOIx4lbrWy0aAODif+oilf0paztPKHv8hbeoM0A9U+VToChydSeI03ZpP/QzuZjCQGWi3cBROw2xBzGSHif2JzBZdRyScOq+FsYcb9+FYxros9YqekG6aj1rJ5hXmG2z/8O+94ca//frf15sdgrVD5druKsNh+iOVE0XBLhqt81bbcsXgm6sesNw1QggDJvS0xa/IhgIVHRXR2BwtUAB6vIwcy6nMZIv8JKjrPZiY5AMqsYtGkCP/4R01U016uMBWxbtQy9uTGP2xoLZZ7VMGiFUhjMAUwPdTvTaXxXIduSOd+wFWTtJM6SkoHKGxPQMRE2EeQwKTAnsJM25kuuiKi9swFYUJbML2jxymFnJxms9IMzSnPNSxgkFkR0JJvWC7ziDv0Q5fpc0cKtWl3fDKoxAsME9xzUQ7Bh4xj6eL/7xutbu38utV76frJ+9Rg4zGJM0hB9BPGgF4ES6BQdKsago6gDSAMUB8QfTHxYFN9BiJzgSnVBUye8tqnG/URao6oOwYkFl9XYJKgpUM8Rg3Pg7Tu2UxWKOhitMYCjJbtZydKZKyLHOZG8IH29C0OUAWZy0g0hVxIU+ea1BcCfqCemCJ5FPOVeLpKYiYJLYe6yh46olH+WS52usRivpwl7NpfeiO8o1B0hIVLnQ4XfKVFtT8YCEVwHI2yufcK+WweCCP/AWpADD6ise9IuHsCgpcKzul7RBYgwBWuPWMJ9FyO785JYQ6FbDIwZlPHgQruSoIQOaD2sQkMJ1fhWsUCWQsE43KBC4RpmkOogoao3XOzW0M0wtWbaD3lXijDC/QPYZggxZnsOJBXe8Q3m75VWII+mTbTWfDOTK5mATVmLTRBgC8RoTaIRodgWbdmUsWlqlCjmD1SsvebvkFrpnBkolSg5RjvZ5GbGhPpklTZrO5rmlloYBqRqsBCreS4cb/5AXwD8Mh8Ef3h/+BWhpKHH04PTvvH+wP+u1OY9XCh4q1ka2W1b82wlcILei1VCb37OgZyQna0GrC9HTBo0mxRAT7GzCKtLn1ga562Y40tVy52Hsa5JLU8QjlfqmzkQlc9EK56I1ruU6NbgJdz80sqrWqVKLpX++kSElo6EP1U435c2N27BEz5cBgLF/+KnbcorHQ1OUQmxS0vpQ6Q9/MhOjPnXQ08hxVSnVxwHS9FudEm7G496fhuCQiUceQZqBYH1W7T0a6ZEErpPjzQihrkAwrOYnm6s0gW3NnS2hHnTVTj1//y+mH3XRKlH4LDa3RNxMSi5W6fn1Do+hVGRmcJVsuGGmqk+ArrQluh2QJRYZ3+IGV7KBEtNEn5wE4clAbk4ZqiaeE2KKdKi/VUxARIiiJ8NgCu24EwRugehbcthgKV4KYi1QYKMUsbVIbRBLM5ZrAQhbxhU091VgWL+f6qhhZvbzG9o97YtfRGiyJ7woFq8va7O0A/0AVzOfiUVV60hpwtF3aMY4VGaeBa5/ueR7x4JWCwWzpWTzOobjK4MdkhjwiDguVA9jli11h0UF3oP42pvHNjUwatyvASsG9RQHzLa0HyzcpVLYmdAojUnmYqPIWmmGwkkAeiBP/BiM1yvsxjDx4XvdbnGvHBRB+lN94sNIaTeW4iCSghEHt3OfnaCWOdUIKzjE/xQdQEwMVHcxNAUUMogBDE2NcR0HLiQyA+7B/mLg4kCscB1q/NJbra+Xr8dg3+cH7eDS4ODv/OweOFOSFae+2nwvPi8x4JC1rJLZ5zYNBcwLWW/TzXkVcqCpikOIuPjlgBBjLioNSy7tyw24mG3P8irInC4wbnG8XBuAROg2rMdfKAM6VM3hzTWknu8MN5OZKo+RAoYjO0tUWMxZvt6lJL9yCU/YHnblGideauUxImzBVpHF7ac7JFk6Z5WBUbprA2kDBYNFCLcmqUJZMoXmo5eAhmzv0K1jPpLjbwkffTQpsi35vMjBJcyD7I0uLluzedFHKmEKp7eQUi5FXcYpVGCgwVxuPcEQeOgT9zEMJkmb+bF7d+KPwoaVMt/gRxQ+Uh5dM8LY13Pj+Y+/7k973A2NvkYeiL/HQ5RRD/xjVbNk4+hLOQRRj3vBcRihIxHCIO0ECdLiy+AKTHPQbBsymgiKtJE8BqWpWnQ2E9mza08pmVQ9lmodLVQQWegGW/5FdjOIH8fMbuqHH68dRaIxGu5n8mrH7sOKVpUUtlUnRvIHFXZyDAPbzLAbsq71UiYRxBIiDPJJREI9JolxGIQanwYbxE9qxM5U3frAQCqnLhgw6o8tddUv42s73k2me3rbskbdU1/2I/k2SGOTs2YB+LE8RBeiBz8A4BdMkaZXdoOVLI6OclA47LHiFnWeTrZ9MQUMAImoRaFQ18KdLlSWIuQAsK4Qj3cO8IsQbZv91x1K/r5gbNVQUTTcXtHDyTBE1lfdy6qWL2QikMt24nPbTe1SN9Lqbz+cy0b2hNxvfoWw+On1/Ntxwud2H+cvdVz8P84ncmWjUyalZlyRvU+XxT6/G8O94Z1yrDOYMTJIv++enHN7HX0enH+wMdhOOt/7OMjj65+dn542D+OvrwF1tcHlw0B8MmivuvKnIbXq88/IlZ23S/JSzkKS2QYaOyPwb9WuGse0bqe5AigQgKNGsLEwXcs+5/GoH6H7Q+gS7wVwbfrU/R3mWqrUDTPmwCGFxRkfHEAdoaJE+CEnF8F5HgoFeq5nu8HY1ynztxi9HQ20SBizOKFxjFbOtsqK1vEt4ZegfcigP25+1gEKPdlPDjaI0kFUUW0DL9wQkvqMftddAOaro39ReKVLia/WzXhstZayOKowUlwHyk6EtDFuz6xMxW46lHIaczKK4/K5anxVcWi3P43lrx3JVwLg8zpXBVc0j3D5ds3EGP7VFZspJpkFGNiBXSZ+uxWNdALWfxKNqHX4pBD2V6c14vaDlUwJq5ibWyph0rvK1ESvhi0g+ENtSmMlAm0vhu3VwCTcr3qjis0Fz1LbXbTy698yYfo5aUrTVXHu1+4OmVzxHGltVUSLSfkuty/YK/dVGr8SkIYl/0l02dLPk2lu7WMZFmM+1aLmFmi25PoY3t1tAxjjB2G9kJIKJGS5HM/SXt1QWE5Fb4j7sGz8Zw9sUs/jYjualmW2elrJWqy2StAU8lnzVWFemdR5gltFaYxhw2TWhbwaxUC9rQ8nLjnVgPATjJ8RF1Dj0b6I4BSOPJkxL4zRGEzbU2CxXI83gKpG9DFhWVRraBz+J1gJW+8loNttoXRuZ2lhYE5Nkva0FHNmbfwg0ZX2sDRsYrqaUblG+Tw8D9pwbQ1acy//gXIZySywtVYwHZERabqYYQyO2Gd+o15WVg+EhS+9cbTGM101jIFPXPZBqDomGu5RSs3yahaQGMAkD+CSeOEa13gjKMJcT/poJU/e5GngYbpyy9RRZCa3QcJyMAfCFRH9eWRXHoBOCCqFc2f5eVTO2PpZXm/Bk81orZX3v0sxFj9qAKFrQFhC2U1HV8JruN69tfQ3PFeNCJUtflJYvKhJ2P1O3bTOknDAr21OLlH1q2KaIy9sQnX6L9chpWTPUj/Zn3sopLGhUntPz3ZoUF3Ee0FI5kyVPptNw1E3kP3KJCan2YxIz1YdzGLRxxks+micxKpQ1XagO5ykf+TIsIdbeMHXLzl1zbt7Mcy+MJjH7W2w5YtHoKErnEgSE4FwRcQua+AHPNkHGPf18dHi0T3ngWMqeg9i8bYAPN7BbSokCcgD9oR4nfFbOAbhP/Jk3G2GpHesF+bATrq37eXL5HjBNcc/AbZdiMB48nudZy54kV8ON6D4ERbmVzkJecG5tATmTxRbAu4fwdphBu5Qd2GEoPJVPqWuwwNoL0vtOFN9K3HwHP/IIE1k3ritLAZSHe3bqRiXHDK857jJFgTHX7zj+Qltf86ybzqchZsV3cIthbWVAtWFJsCdeVX2mQJorkxrX5JeD4lc7186iBUWwJAY9uPTudTV0yMU1ncp2X16v6dYyZAo2ZjhdI9DBgeffg2GDpkqD+5XOvEgRB6ef+6cH4lbeB9u3L9++ZnMZ9yog+6rjgMgSggkHBpASFc3+rOU8BaOeTGZzeaP4QfeGtC/J3UHnG0iFPaOZw/7n08vjY5dMReB53Cjviejsr8IBVV6sxG813SeYYr7TRxKUPA0dTob9HIzaeIY7Q3Can1P6aKLztCN0DZZBqDSEW1+dGsUSGHWjBCLdKf/BYZgGMc4cceID3Pehr49VOuSIJUAVzlKMn2WxeGnmalOap9p+oqYTIzmlZgrJRHRnL0aijAnMAMF44FjvD0qt05hMCBx+DlxRH5ydHlyen/dPL1RSPSYS6wUjOXpAnoLJ57VSOZ10FDd54Xh5nhReWKGrBTI7Lg35bJTDtCZ8g1xo1rGnclc5JEEusHxmhkT/ZPnmQqZTv3yhpQvx1img1DQkqK9iPNB5+RvKT/SGMe+xAFDvk39qSIqstY58ZCRUMk+K1qM9zE09SrBB2rU2eLOZB0q2KbRmFVdca0UIK0U4ddpLp3GW9lhJgugr1W7V/saeHnd61FtH7PKPpyWNNudfW1UmtJTzmPO9AODFDHAFERkCk2nsZ9R9tTvOosRs9sqBYWSoaR852wvIDdVtpMQilW3h7Q6JpsLAsuxB0wXigKDu9iin0FjJBq8kJTVBgt4iqTF79KrBH6eC64mR9AMVZ+cgjprfZmDLkvMk69WhZFbeGV51HDm2QyKeLMzq2KNrh2Z9a2S7s6rPxo2YTT0v2SHq6L/svrI2oUwPhZuVea7uU9IaT5cpHVwUmkm+wadFrONTrhMl8+G2iNak4VAa4CdMrsTMH95NWPLZsvPAsBolhwJB4Hfl9I/bLJs3H62Dl9K7ULWbsKDlQyHqNdY8JMaZxauzBwgjEk9sQ+MzkF4Wc+qHUkwAfJanekfkcIO2a1rwK6cMxweCGNYsaOZuTRIp9V4Uc4Y5FOnSiabIVXPMF1oFzXtb59T86IW+KMoWT2plzTXHErVZq2csQqrVSmu4VovRS5XoVz2+ADznFft0K8GAWulE3mDEHpDq1Qo7IwffIMbMfX/PFWPlXvHnirA1el0hwoy+m8UXKr1GCbbaiZ85VVa1xgM552vnDlJd0z6ymPrJnXxJ7RWCkVum0X6DdCSROM5n8xZOOoClg/shQZHvvfx2jz/+JoMXJZ+nHB9KuMhoPI8peElWEfbaazSXtClsaXe3SxMv5XynuKYfiY8XF5/Ep7PBBRqhH/oXQkHCyBsl8UMKgorX4RTH9O/jEI8ciPPxZIrrhBFu+lB7nWtSSvn/qpal0w9oW5esZYDsj5XKT48aPZYBXGdCGImnpKTdNF56RDUJyji6hFFv7d/QETroJTmJ/xlOp/42nhzQ+hLi3sFUnF6I3Z3uzi8CHrx9/Yv4+vZ1G2PGU/lFjn4Ls+03r/7affVWtH77eHFy3BHT8E6KD7DYjdvi4BYsKrlNGy3xPzHwJ34SqiqWI6eEaj9AvmKIfOhH+Uq3eQY4q/zXFq8xt44O6wrBrvBk32JUA7EXcu49bZut92AguZgmKc2Tts7zKCdZvboiAyiDA7aNty4Wc6kT2upjrFEYytm+w+65mkrwmKfOnoaxo7vbU38rABHLV1qDW5IgcN8hrRHn2d6rHZIXiUznDpTw6Vjoa0LNhYH6lzuubG68RvF4wQdOzfFIwHGryH5ZhjUeO02awlxLW9hWPc/FdKF2cbZzJKaSdq3gRtXghzMPN9ION/RRgIU4qkOi8l0pRgULzv7xpz7pIVx9omR5xG0jY/mE7qFy3mICMg4XYK/nIDulgWswl+fH3z4WMkU5k2BMbp+iBNqnsq3X3s8bMzBxBFZcsVPqTxq22jFOo11Hdq5Ksi8w9AeGz0HUyvjLHmsDX3PEDWa4NuA8nKNqYVsuag3vJV56LctVtEuytLBpnhd+M+LT/U9HxBGcQv6O7PKaMgMgvLm/wNn2/43l7bR+FYlVQkKFxkXKWSB3hUZycUYsoTWI42SMZ6PEhffJ0N/tbrdbiX2l2hdnW0OwwPPn4bYyV7c1E/BGs4IWdjJPQpmodOq8Wp7qtUSbc6bKtYCVkYYXmGbhDe8cKavrhwrPnQq5KkytUKcTISq4UzkPuJ1iIUpAGlAHs3bzUXf/tPmdhTTjAJhK3zXBNNzgs8eY1blbdcySytf9RaQLWOSOaUJUTpunpehIzwgru4c9cfUVMiF8r1gU12Y3OwnL2Z0CFOViThnBMzxOIkjXM3ydEoCNXDpQYp7ENwlmmyBrlMc7gT2LWK/N9HKWKyjQOjadgqrAlTXrrzUXOwysdXncWNJ2RJ3Bl2Ncla/wfad0T9hTxV1cTZMyw9fNZU75X8qGn1+Jj3ooPT4KT7x4NHvcJEcsIOnrZvsJxIT1bh5kmx0BdnG7192dPH3fFv8h7BKTeYoloCr8cjEZbaVYKQ3iO4OnOcyujQaH03TVsupTPJ2qkAwum+m4ryLSpk4q4wmN0ZMa2xXGCS18LL4owfr3grv2qkvhvxRSae/RjJJ38dAI9lwZguvJ1m0N/KlhWiVlb/1UmVpVo+F3DneURX+PRzRfkHx20QDZwSpbMAkW39qtlH+hzubHxMoy+KjSWkDEssuf38mvPohAaq/SqYoEeHmUhVONBXe0gKBq4Yg6DGybubRmWxkOJvFvlS4cRhXHUiQdgsfMZ/bhWnY5bCdjdq5U3W9gCe+T9afnpsa5MS+RQOJHsfu0Xb6l2L1CC07ceCL+Fo/M+kxemtSbRr0wm0pocRNKw7vNdpM9mMjVXlkdiCznaUdwt5QX1RHFGNQ9mLgMs9o0xmnimAqKO2/BFOpx8Af3jHZ3lqiUfIRHaqmprMBARYoID+QaCuUPmY165vTUaGsFjOnSK5FQK2ajA8vaT2oVTFRhceveZUKuq+8sSrp13noKR75WByICS9csKWDSQLF5gZQnk3wt5mHG6lO7ap8+Q6H45KNRpxubCxhjbq9k2GYGVGNMreEB16n2hc8KDM81owHTGeHwXm/MUk4J3kKdVRLE/s9z6h/iJRPH38JKL3d9tQ8H1jXUVCkdCwhJHpZST3FMFc9/gIHUKKoMVJxWxpDQ5kWnt5h99B7Hiwtp18xRh/p0O5H4D4pr7IgUBX9rc6pYO0Pr0p+h0lNLBlhKoNewRMW3+4SNQ6fwanQK65Dg5jboFreDeLWzUHsHH6v+X/LsgA334dMl51RsVy2v4cbT0sOM1nYtvn7zDNfid8tci24MWgBx0LjKLxQmGamYCOqFprhI5WCJePQ7cvectW01elz5Wpr1bTGjc/fJGI5iv+rDHfBax+G1bO4THzPrK27W041T3GF5Qp6LpY6sIiMMn1F2PPm/ecKquYqiqVdJFO3gkZOZVgVogTRP1vK+/1UGdHCyTtjaQs2DZzJjfhYn7RWHItOO0+IgWyRYoRINwZwqHQEMHsLaLygXymmHj63jk54EngCf3kI33dndvRW5NnO/XFDj2fN8ggJje48wcsWH0psWieGYQm5y1qlaMdfmZJ94weRGlVWCVh9d76m8XqD9ozltWZaX7WsdZrRbqoGilKHIrk3nCD9X2Vi10+sqiLB0FDI5yuLqR1ZcaUDAvGFEaVCPiouevMeya5tZuTInYK7VNGJ4Rdtl62hTTGjTaTGETtlldVXoON7teWkvxjlvfzQHpHxHn2V8L7PglrShSnPsiYtkYfvF6ERH0SIxSobUduAHt7gFgQ7HIe8arA2Lz7TA3BgbTjiWMibu8mhcHtO4VxXb9F0uPk0VMQzz+ka2Xq/GaZXN3OcOVQu5Tx+qwWjn+unLOPGlxOp73KWjMlrp5Ljik6KC8QZzE3THLZ4oAqIMt8aQJPIpZbUobDfr5o1qhp7eONg845rRR201JSUZJ7xxObvYN33YjdwG6VTKeWu3a+8pBehsCqyVhVE7/MnGQqcUNyvAd6lUvJZEw64GIDVEITuu1dQh9aM8My3Qp+1faJrQ8YBqqhhzpBos0FfTjKnZmZWC7hNQSyxggeoEMd6CiV6dGytcLi9fTV41LSP0qqw0nLv4gR4YNj5d0wB3YYj7kXOXJxFfbeuFy7axIK6G18cguO/0ysu1/tC9NHORogG19AeZxzyfQAziHAVKbVHPaMUuFzLD8y1lmoZ0uk+kz25NGGlqiWdyHR9K18BvS5yCVy21xi3NgzbGAkwf4I+Cj9R2Nms7/mptNbr/HLb9xJuZBw06OHhbtPBMcHU0uDt81cjLf91508TLrUfsm9z14uRdm5yw4b2KKdV1X8nU31XsCPrm6rs8nI7F+/e4FcSwY4N4NgMVZtisMC46WJuP/FZ2n15ZGy85DhLlUztJNUgmtWrwzAMtjjVemRle9x7CENTK044Kfse9fOy+fPsGEP2x//nAPoMw9WgPDa7z4S9vOtHN0q6BfaxhPC3rooOdtqrg1iHHNp6K6FNd1S2Eoh3HfLz3YKHFWY8bW0Hvnodj7pKhTTl0gKf6uNj8tXoY/IPLYwIBYLD6FWRHvLK5R1gmfyX8cMvJxGy4BsK9TBYTP80sS93R37cN3tg79L9z8G++ZfAGX8/G3K61mwq3sI9DvdduYSHshfpClfh8vn9SbgOidCSUkjiNTz8f9g965adPUoGbD4siozjLpjKSgX1qdRPSAUbQvxluqkUE3D7QCpaBC/Kxj5+9tkZklqYthoVYMwvi+SumVKAP2vDEXw7APWPaqNvYv9XMD0zHmotzy69n5eFTzKD0Innj0xaWLNVbCe6k90+ZxK46On6t2C6cy95urVyx1iofX9snqOaJl5IAo2BHy1gbq1fYPn0iql0Rya+64tjPI1gXKZmstt2xI4li7GVYXX2+oeZ3bVQrb/1d/E4af2vY8FpMmpQNrodsB2xr87EYjgo2YZSJEpGa9z+Jlm1m1HJTcJj2dsVP5DUDbuioE8DMTYifjj71l2xOtLYwjvIJ6ua9XQvVnNg+ouN4bBvCYMU8Sbx5oKJVleeTeep8zt9n2aNNtkPrvBj312H5e1gRYUAdpuWy4LDUHv2pb7zVF+6k5QLGjg+Ac+9bP6ZsDJQ5mZtX+3ix2avdaw1PQwIl9cIuHMogQItNt+uu8Yc/PsznYtUwQZRZgguLfMtG2oBKHiQ1YAyT79eFET1ByIZe/u1Uw2RY2vXsGsMSKjG4JLegiW2x6+3s7OD/LoPaqsRzBHRV6+efuzsdMfO/tnbwR6toc5se7uJDJQPbbbKNlRBsal9hFXowcIp3fxKzGTXWpBxef8r3saP4oUnymBfwSQuLblkyi3bKY1r8kq5JHleSvhxp+Oa1MqJYr7BWLLxejSjaK+i5sjzRs1dQc2V5Rc2eQcvVdYrtUGZiVnOtpyWMuyIGiqq4tvZucl8oq6QnHjW6OF0Ls7U0QnrdnQklaYn/5s9d6dI0dPcZ1AWstjIEblsisDh8qmHaA/aqGXz6MoTDbrMYqU5xR1FSjg8+CLIlJ0y6XGdU8S6cTquzS5GHT1Za5RMpu1CfxGwglOYZd/zqX+DaeL5bwxVEwwvzFBFX/J7Mwu/ws3TL/HgYbmj249HbZj+eE/N8vmUTekuLVbk1n2l0iisAH4+bGUugUGW4eEjR/0sE06cPaogQGSod2nDbLEmxw+MUHb2rg2btrl53xQG6k8uTKdglaMYkzbhNWXfZGYOViOOSQwar7vWCwzrVsKUdKlOpUw5PXbVas79uear5cmed5VEjzKH/TufCFT685y6exMBHj18ZCK569t50OUtrUXNnV3jDzvCrcUMNvZ0Cp5Ue33b5YwJG/BvDaHGCn0I1vptDZHlWBNNt7K4Z11Qs2BDBfOY+UrwaUuv1JwqJRtMYppbav26kHDh2rXMeLR+owcFHga5ymEZ0ShWSvQV2s3hJv/nj0fCmXU/2GU+BhmFAn3A0ZM7DLaYYWUeIOCMHy7OEiAxlyOyV6/MUxbc3KtTDb1Cq2IaR8+3I5MWUDQedGwZWDBD9Iu6jSprSaaqFrxSRrlWyQUP0YMkGjNJWdEU01AGcCglWtgp2VzDJnvrbtMX6zxulmef7x7pBn6MfLcrvdsJNKzU3s/JHtWjLW7Xh7r0/zfGjHm5fBn+Kymx/6VjX2B/T2I2iUdNy22D9WhI6Xg1+50rdne6bqvxx1zN43v21F4w7GPMCyqAZZjz6HkwPl+2lr2WLk5r1ewTt9sQXsL71V4HKvRbmKVdNIb2K5CiFJkgk3vxRE4yktdQuA0kZWCGlVsVzWKugexQhsY7bKr/uzqKyJiGbfaPkGwG9//anutZfkeAvf9pR2RbYvzoICk+kYjcrLcLs7SDVTKElvcB7pSCODnU7ZirkM9oB6HrLTYzqS7V1Ed+dvKNYRftZPXJm3Ecd3tjXSXOoZcXj5t9luqnCZ/WD8uh40M3TWLQw7PFe5fa0N583ZuOr3XrszqPRnhT7gubFs1v9SMZ5Ol1oXfusPg/pfOLj+EYlyatDVnHE/DmZQ9H6zF9YadP4VYPGR1p48IdHA1X8RB/VnNIpyOtjwWLritn3PGH5grnZnnDmZCtwVfq6yaRxHR9LRk2RW9VMlpoZaJ08xp9eUgmre5yzXjXBOgLDRXstfNBp4yEFYAVGrlNRMnbCVq0SNQh9Im1mIWWlZUU5bvoAQUEbATj3qpZhcnl+/I1GWaVEkUTiOJrMrWNXd7GcybQwDODfye5OMV69O+SgHKaehMYW5UanlFomOzajP0/7Nm/VVQlDplar0UWdG4UL6/pJUWiYF4eS1eJZtcEsmXEPDAN+IEKNsrEpQ5VWbQl9NRniL8SnWr4xf17Hx68DkfmGWYscexXkoeANLPglLjXrQSxqW5dnvt3H8+xHVab48HxWHuHWbCrSUsG0L6snwE6sdp35lZjjib2sKFjQbn0Xuuk+97PKPszqmVN2RcuNXtQ1ni6vrtzpRUW6502TS+spt3pRj+7XqFe414ua6klHRT5XVH+Gp73qZW+Y725SWT2xbqu2ZyaDci6oMk49dWrkDASD+lqc+vILf1XPOIuz+qk7/JwL7zFGdlOGIVuK5RGzhoWqnUyVD+kZ3Smgldrds07C1VLRdpWoI4LRylbvlb/jN7kYxWCZHUUga5J8bgacl9kTpdk73tVjUhZqqJvirxXmKbQhBrd5RtKV/I0VCamgWyIX48kE4w3KGMFPnJC56nkcgfCILp5XRB+qFINqG0//C5N9bQ4="
try:
    raw_json = zlib.decompress(base64.b64decode(bundle_b64.encode('ascii'))).decode('utf-8')
    files_dict = json.loads(raw_json)
    for fname, content in files_dict.items():
        fpath = os.path.join(workspace_dir, fname)
        with open(fpath, "w", encoding="utf-8") as f:
            f.write(content)
    print(f"✅ Deployed {len(files_dict)} helper modules into {workspace_dir}")
except Exception as e:
    print(f"⚠️ Notice deploying bundle: {e}")

print("🎉 Setup complete! Helper workspace ready.")


In [ ]:
#@title 🚀 Step 2: Start Autonomous GPU Worker Loop
#@markdown Launches the helper node. It auto-discovers the Main coordinator and begins encoding video chunks.
enable_debug_logs = True #@param {type:"boolean"}

import time
import logger
logger.set_debug(enable_debug_logs)
import helper_worker

print("=" * 72)
print("🟢 Launching Autonomous GPU Worker Node...")
print(f"   Debug Log Notifications: {'ENABLED (Verbose)' if enable_debug_logs else 'DISABLED (Quiet)'}")
print("   Keep this cell running to provide GPU compute.")
print("=" * 72 + "\n")

try:
    helper_worker.run_helper_main(enable_debug_logs=enable_debug_logs)
except KeyboardInterrupt:
    print("\n🛑 Worker stopped by user.")
